# 01 — Pipeline Data (Data Engineering)
**Projek:** Analisis Perbandingan Jumlah dan Intensitas Titik Panas antar Pulau di Indonesia pada Tahun El Niño 2023 dan Tahun Non-El Niño (2021–2025)
**Kelompok 10 – 3TI B**

Alur: **Ingestion → Validasi Kualitas → Cleaning (filter) → Transformasi → Agregasi → Simpan dataset siap analisis**

File input: `3TIB_10_TitikPanasAsli.xlsx` (sheet `data`)

In [3]:
# Library
import pandas as pd
import numpy as np
import geopandas as gpd

pd.set_option("display.max_columns", None)
FILE_INPUT = "3TIB_10_TitikPanas.xlsx"

## 1. Ingestion — membaca data mentah
Membaca sheet `data` (579.175 baris). Proses ini butuh ±1–3 menit karena datanya besar.

In [4]:
df_mentah = pd.read_excel(FILE_INPUT, sheet_name="data")
print("Ukuran data mentah:", df_mentah.shape)
print(df_mentah.dtypes)
df_mentah.head()

Ukuran data mentah: (579175, 15)
latitude             float64
longitude            float64
brightness           float64
scan                 float64
track                float64
acq_date      datetime64[ns]
acq_time               int64
satellite             object
instrument            object
confidence            object
version                int64
bright_t31           float64
frp                  float64
daynight              object
type                   int64
dtype: object


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,-1.45464,127.43291,341.74,0.66,0.73,2021-01-01,404,SNPP,SNPP,n,2,287.14,7.48,D,3
1,-3.98923,122.10133,338.53,0.50,0.49,2021-01-01,544,SNPP,SNPP,n,2,289.81,4.83,D,0
2,-0.92415,120.00077,331.21,0.40,0.44,2021-01-01,545,SNPP,SNPP,n,2,291.39,2.59,D,0
3,-2.15948,115.58220,328.71,0.39,0.36,2021-01-01,545,SNPP,SNPP,n,2,286.21,2.28,D,2
4,-6.04117,105.93034,333.22,0.42,0.61,2021-01-01,545,SNPP,SNPP,n,2,286.28,4.17,D,0


## 2. Validasi kualitas data
Mengecek missing value, duplikat, kolom konstan, rentang tanggal, dan tanggal tanpa deteksi (gangguan satelit).

In [5]:
print("Missing value per kolom:")
print(df_mentah.isna().sum())
print("\nJumlah baris duplikat:", df_mentah.duplicated().sum())

print("\nKolom bernilai konstan (tidak bisa dianalisis):")
for kolom in df_mentah.columns:
    if df_mentah[kolom].nunique() == 1:
        print(" -", kolom, "=", df_mentah[kolom].iloc[0])

df_mentah["acq_date"] = pd.to_datetime(df_mentah["acq_date"])
print("\nRentang tanggal:", df_mentah["acq_date"].min().date(), "s.d.", df_mentah["acq_date"].max().date())

print("\nJumlah baris per type (0=vegetasi, 1=gunung api, 2=statis darat, 3=lepas pantai):")
print(df_mentah["type"].value_counts().sort_index())

Missing value per kolom:
latitude      0
longitude     0
brightness    0
scan          0
track         0
acq_date      0
acq_time      0
satellite     0
instrument    0
confidence    0
version       0
bright_t31    0
frp           0
daynight      0
type          0
dtype: int64

Jumlah baris duplikat: 0

Kolom bernilai konstan (tidak bisa dianalisis):
 - satellite = SNPP
 - instrument = SNPP
 - version = 2

Rentang tanggal: 2021-01-01 s.d. 2025-12-31

Jumlah baris per type (0=vegetasi, 1=gunung api, 2=statis darat, 3=lepas pantai):
type
0    515450
1     14765
2     45963
3      2997
Name: count, dtype: int64


In [6]:
# Tanggal tanpa satu pun deteksi di seluruh Indonesia -> diduga gangguan satelit
semua_tanggal = pd.date_range("2021-01-01", "2025-12-31", freq="D")
tanggal_ada_data = set(df_mentah["acq_date"].unique())
tanggal_kosong = [t for t in semua_tanggal if t not in tanggal_ada_data]
print("Jumlah hari dalam periode :", len(semua_tanggal))
print("Hari tanpa deteksi        :", len(tanggal_kosong))
print([t.strftime("%Y-%m-%d") for t in tanggal_kosong])

Jumlah hari dalam periode : 1826
Hari tanpa deteksi        : 38
['2022-07-27', '2022-07-28', '2022-07-29', '2022-07-30', '2022-07-31', '2022-08-01', '2022-08-02', '2022-08-03', '2022-08-04', '2022-08-05', '2022-08-06', '2022-08-07', '2022-08-08', '2022-08-09', '2022-08-10', '2023-07-26', '2023-07-27', '2024-05-30', '2024-05-31', '2024-06-01', '2024-06-02', '2024-06-03', '2024-07-10', '2024-07-11', '2024-07-12', '2024-07-13', '2024-07-14', '2024-07-15', '2024-07-24', '2024-07-25', '2024-07-26', '2024-07-27', '2024-07-28', '2024-09-26', '2024-11-03', '2024-11-04', '2024-11-05', '2025-11-12']


## 3. Cleaning — memilih kebakaran vegetasi saja
Type 1–3 (gunung api, industri/flare gas, anjungan lepas pantai) bukan karhutla, sehingga dikeluarkan dari analisis. Data mentah tetap utuh di file sumber.

In [7]:
df = df_mentah[df_mentah["type"] == 0].copy()
print("Baris sebelum filter :", len(df_mentah))
print("Baris sesudah filter :", len(df))
print("Baris dikeluarkan    :", len(df_mentah) - len(df))

Baris sebelum filter : 579175
Baris sesudah filter : 515450
Baris dikeluarkan    : 63725


## 4. Transformasi — menentukan pulau dan kelompok tahun
Pulau ditentukan dengan mencocokkan koordinat titik ke batas provinsi (Natural Earth Admin-1). Titik di pesisir dicocokkan ke provinsi terdekat (maks. 20 km).

In [8]:
URL_BATAS = ("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
             "master/geojson/ne_10m_admin_1_states_provinces.geojson")
batas = gpd.read_file(URL_BATAS)
batas = batas[batas["admin"] == "Indonesia"][["name", "geometry"]]
print("Jumlah provinsi:", len(batas))

PROVINSI_KE_PULAU = {
    "Sumatera": ["Aceh", "Sumatera Utara", "Sumatera Barat", "Riau", "Kepulauan Riau",
                 "Jambi", "Sumatera Selatan", "Bengkulu", "Lampung", "Bangka-Belitung"],
    "Jawa": ["Jakarta Raya", "Banten", "Jawa Barat", "Jawa Tengah", "Yogyakarta", "Jawa Timur"],
    "Kalimantan": ["Kalimantan Barat", "Kalimantan Tengah", "Kalimantan Selatan", "Kalimantan Timur"],
    "Sulawesi": ["Sulawesi Utara", "Gorontalo", "Sulawesi Tengah", "Sulawesi Barat",
                 "Sulawesi Selatan", "Sulawesi Tenggara"],
    "Bali-Nusa Tenggara": ["Bali", "Nusa Tenggara Barat", "Nusa Tenggara Timur"],
    "Maluku": ["Maluku", "Maluku Utara"],
    "Papua": ["Papua", "Papua Barat"],
}
peta = {prov: pulau for pulau, daftar in PROVINSI_KE_PULAU.items() for prov in daftar}
batas["pulau"] = batas["name"].map(peta)
print("Provinsi belum terpetakan:", batas["pulau"].isna().sum())

# simpan batas wilayah
batas.to_file("batas_provinsi_indonesia.geojson", driver="GeoJSON")

Jumlah provinsi: 33
Provinsi belum terpetakan: 0


In [9]:
titik = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["longitude"], df["latitude"]), crs="EPSG:4326")
hasil = gpd.sjoin_nearest(titik.to_crs(3395), batas.to_crs(3395)[["pulau", "geometry"]],
                          how="left", max_distance=20000)
hasil = hasil[~hasil.index.duplicated()]          # jaga-jaga jika 1 titik cocok ke 2 provinsi
df["pulau"] = hasil["pulau"]

df["tahun"] = df["acq_date"].dt.year
df["bulan"] = df["acq_date"].dt.month
df["kelompok_tahun"] = np.where(df["tahun"] == 2023, "El Nino 2023", "Non-El Nino")

print("Titik tanpa pulau (missing):", df["pulau"].isna().sum())
print(df["pulau"].value_counts())

Titik tanpa pulau (missing): 7
pulau
Kalimantan            182366
Sumatera              115474
Bali-Nusa Tenggara     95448
Sulawesi               38436
Papua                  37073
Jawa                   33002
Maluku                 13644
Name: count, dtype: int64


## 5. Agregasi — jumlah titik panas per pulau per hari
Hari tanpa titik panas di suatu pulau diisi **0**. Hari gangguan satelit diisi **kosong (NaN)**, bukan 0, agar rata-rata tidak bias.

In [10]:
DAFTAR_PULAU = list(PROVINSI_KE_PULAU.keys())
kerangka = pd.MultiIndex.from_product([semua_tanggal, DAFTAR_PULAU], names=["tanggal", "pulau"])

harian = (df.dropna(subset=["pulau"])
            .groupby(["acq_date", "pulau"]).size()
            .rename("jumlah_titik_panas"))
harian.index.names = ["tanggal", "pulau"]
harian = harian.reindex(kerangka, fill_value=0).reset_index()

harian["data_satelit_kosong"] = np.where(harian["tanggal"].isin(tanggal_kosong), "Ya", "Tidak")
harian.loc[harian["data_satelit_kosong"] == "Ya", "jumlah_titik_panas"] = np.nan
harian["tahun"] = harian["tanggal"].dt.year
harian["bulan"] = harian["tanggal"].dt.month
harian["kelompok_tahun"] = np.where(harian["tahun"] == 2023, "El Nino 2023", "Non-El Nino")

print("Ukuran tabel harian:", harian.shape)
print("Sel missing (gangguan satelit):", harian["jumlah_titik_panas"].isna().sum())
print("Total titik panas:", int(harian["jumlah_titik_panas"].sum()))
harian.head(10)

Ukuran tabel harian: (12782, 7)
Sel missing (gangguan satelit): 266
Total titik panas: 515443


,tanggal,pulau,jumlah_titik_panas,data_satelit_kosong,tahun,bulan,kelompok_tahun
0,2021-01-01,Sumatera,0.0,Tidak,2021,1,Non-El Nino
1,2021-01-01,Jawa,1.0,Tidak,2021,1,Non-El Nino
2,2021-01-01,Kalimantan,0.0,Tidak,2021,1,Non-El Nino
3,2021-01-01,Sulawesi,6.0,Tidak,2021,1,Non-El Nino
4,2021-01-01,Bali-Nusa Tenggara,0.0,Tidak,2021,1,Non-El Nino
5,2021-01-01,Maluku,0.0,Tidak,2021,1,Non-El Nino
6,2021-01-01,Papua,0.0,Tidak,2021,1,Non-El Nino
7,2021-01-02,Sumatera,1.0,Tidak,2021,1,Non-El Nino
8,2021-01-02,Jawa,1.0,Tidak,2021,1,Non-El Nino
9,2021-01-02,Kalimantan,0.0,Tidak,2021,1,Non-El Nino


## 6. Simpan dataset siap analisis

In [11]:
kolom_simpan = ["latitude", "longitude", "brightness", "bright_t31", "frp", "confidence",
                "daynight", "acq_date", "acq_time", "tahun", "bulan", "pulau", "kelompok_tahun"]
df[kolom_simpan].to_csv("data_analisis.csv", index=False)
harian.to_csv("harian_pulau.csv", index=False)

ringkasan = pd.DataFrame({
    "tahap": ["Data mentah", "Filter type = 0", "Titik tanpa pulau", "Hari gangguan satelit",
              "Baris tabel harian (7 pulau x 1.826 hari)"],
    "jumlah": [len(df_mentah), len(df), int(df["pulau"].isna().sum()), len(tanggal_kosong), len(harian)],
})
ringkasan.to_csv("ringkasan_pipeline.csv", index=False)
ringkasan

,tahap,jumlah
0,Data mentah,579175
1,Filter type = 0,515450
2,Titik tanpa pulau,7
3,Hari gangguan satelit,38
4,Baris tabel harian (7 pulau x 1.826 hari),12782
